# Phase 9 — Model-Ready Preprocessing & Pipeline Construction
**Topic:** Predicting Road Traffic Collision Severity and Identifying High-Risk Locations and Times  
**Dataset:** UK DfT STATS19 Road Safety Data  
**Pipeline Tasks:** Tasks 29, 30, 31, 32, 33, and 34  
**Deliverables:** Stratified Train/Test Partitions, Imputation, One-Hot Encoding, StandardScaler, Leakage-Safe Model Matrices (`X_train`, `X_test`, `y_train`, `y_test`)

---
### Objective
Transform selected features into model-ready numerical matrices without leaking test or validation information into training.

## Task 29 — Train / Validation / Test Split (Stratified)
Split the dataset into **80% Training** and **20% Testing** partitions using **Stratified Sampling** on `collision_severity` to guarantee identical class proportions in both sets.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

data_path = Path("C:/Users/ACER/Downloads/New folder/cleaned_datasets")

df = pd.read_csv(data_path / "master_dataset_v4_selected.csv", low_memory=False)
if "severity_name" in df.columns:
    df = df.drop(columns=["severity_name"])

# Separate targets and predictors
y_class = df["collision_severity"]
y_reg = df["number_of_casualties"]

# Feature matrix X: exclude targets and identifier
drop_cols = ["collision_severity", "number_of_casualties", "collision_index", "collision_year"]
X = df.drop(columns=[c for c in drop_cols if c in df.columns])

print(f"Predictor matrix X shape: {X.shape}")
print(f"Classification target y_class shape: {y_class.shape}")

# Stratified Train-Test Split (80/20)
X_train, X_test, y_train_class, y_test_class = train_test_split(
    X, y_class, test_size=0.20, random_state=42, stratify=y_class
)

# Align regression target with identical train/test indices
y_train_reg = y_reg.loc[X_train.index]
y_test_reg = y_reg.loc[X_test.index]

print(f"\nTraining Set Size: {len(X_train):,} rows")
print(f"Testing Set Size:  {len(X_test):,} rows")

print("\nVerifying Stratification Equality:")
print("Train Class %:", (y_train_class.value_counts(normalize=True)*100).round(2).to_dict())
print("Test Class %: ", (y_test_class.value_counts(normalize=True)*100).round(2).to_dict())

Predictor matrix X shape: (513801, 39)
Classification target y_class shape: (513801,)

Training Set Size: 411,040 rows
Testing Set Size:  102,761 rows

Verifying Stratification Equality:
Train Class %: {3: 75.8, 2: 22.74, 1: 1.47}
Test Class %:  {3: 75.79, 2: 22.74, 1: 1.47}


## Task 30 & 31 — Missing-Value Imputation (Fitted ONLY on Training Data)
- Convert sentinel `-1` codes into `NaN`.
- Impute numerical predictors with the **Median** computed strictly from `X_train`.
- Impute categorical predictors with the **Mode** computed strictly from `X_train`.

In [2]:
# Replace sentinel -1 with NaN
sentinel_cols = ["junction_detail", "junction_control", "second_road_class", "light_conditions", 
                 "weather_conditions", "road_surface_conditions", "special_conditions_at_site", 
                 "carriageway_hazards", "speed_limit"]

for c in sentinel_cols:
    if c in X_train.columns:
        X_train[c] = X_train[c].replace(-1, np.nan)
        X_test[c] = X_test[c].replace(-1, np.nan)

# Numerical imputation: Median from X_train
num_features = ["speed_limit", "number_of_vehicles", "hour_of_day", "latitude", "longitude", 
                "min_driver_age", "max_driver_age", "avg_driver_age", "vehicle_type_diversity", "month"]
num_features = [c for c in num_features if c in X_train.columns]

num_medians = {}
for c in num_features:
    median_val = X_train[c].median()
    num_medians[c] = median_val
    X_train[c] = X_train[c].fillna(median_val)
    X_test[c] = X_test[c].fillna(median_val)

# Categorical imputation: Mode from X_train
cat_features = ["road_type", "urban_or_rural_area", "time_of_day_bucket", "season", 
                "weather_conditions", "light_conditions", "road_surface_conditions", "junction_detail"]
cat_features = [c for c in cat_features if c in X_train.columns]

cat_modes = {}
for c in cat_features:
    mode_val = X_train[c].mode()[0]
    cat_modes[c] = mode_val
    X_train[c] = X_train[c].fillna(mode_val)
    X_test[c] = X_test[c].fillna(mode_val)

print("Imputation completed successfully with statistics learned strictly from Training data.")
print("Missing values in X_train:", X_train[num_features + cat_features].isna().sum().sum())
print("Missing values in X_test :", X_test[num_features + cat_features].isna().sum().sum())

Imputation completed successfully with statistics learned strictly from Training data.
Missing values in X_train: 0
Missing values in X_test : 0


## Task 32 — Categorical Encoding (One-Hot Encoding)
Apply `OneHotEncoder(handle_unknown='ignore')` fitted strictly on `X_train` to convert categorical variables into binary indicator features.

In [3]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

X_train_cat = pd.DataFrame(
    encoder.fit_transform(X_train[cat_features]),
    columns=encoder.get_feature_names_out(cat_features),
    index=X_train.index
)

X_test_cat = pd.DataFrame(
    encoder.transform(X_test[cat_features]),
    columns=encoder.get_feature_names_out(cat_features),
    index=X_test.index
)

print(f"Categorical features encoded: {X_train_cat.shape[1]} binary dummy columns generated.")

Categorical features encoded: 46 binary dummy columns generated.


## Task 33 — Numerical Feature Scaling (StandardScaler)
Standardize continuous predictors ($z = \\frac{x - \\mu}{\\sigma}$) using `StandardScaler` fitted strictly on `X_train`.

In [4]:
scaler = StandardScaler()

X_train_num = pd.DataFrame(
    scaler.fit_transform(X_train[num_features]),
    columns=num_features,
    index=X_train.index
)

X_test_num = pd.DataFrame(
    scaler.transform(X_test[num_features]),
    columns=num_features,
    index=X_test.index
)

print(f"Numerical features scaled: {X_train_num.shape[1]} columns standardized.")

Numerical features scaled: 10 columns standardized.


## Task 34 — Assemble Model-Ready Datasets & Integrity Check
Assemble scaled numerical, one-hot encoded categorical, and binary vehicle/temporal flags into final matrices.

In [6]:
# Remaining binary flags (vehicle types, rush hour, weekend)
flag_cols = [c for c in X_train.columns if c.startswith("has_") or c in ["is_weekend", "trunk_road_flag"]]

X_train_final = pd.concat([X_train_num, X_train_cat, X_train[flag_cols]], axis=1)
X_test_final = pd.concat([X_test_num, X_test_cat, X_test[flag_cols]], axis=1)

print("="*60)
print("FINAL MODEL-READY MATRICES VERIFICATION")
print("="*60)
print(f"X_train shape: {X_train_final.shape}")
print(f"X_test shape : {X_test_final.shape}")
print(f"Columns match perfectly: {list(X_train_final.columns) == list(X_test_final.columns)}")
print(f"No NaNs in X_train: {X_train_final.isna().sum().sum() == 0}")
print(f"No NaNs in X_test : {X_test_final.isna().sum().sum() == 0}")

# Save Phase 9 Outputs
X_train_final.to_csv(data_path / "X_train_model_ready.csv", index=False)
X_test_final.to_csv(data_path / "X_test_model_ready.csv", index=False)
y_train_class.to_csv(data_path / "y_train_classification.csv", index=False)
y_test_class.to_csv(data_path / "y_test_classification.csv", index=False)
y_train_reg.to_csv(data_path / "y_train_regression.csv", index=False)
y_test_reg.to_csv(data_path / "y_test_regression.csv", index=False)

print("\nSaved files:")
print("  - X_train_model_ready.csv")
print("  - X_test_model_ready.csv")
print("  - y_train_classification.csv")
print("  - y_test_classification.csv")
print("  - y_train_regression.csv")
print("  - y_test_regression.csv")

print("\n" + "="*70)
print("PHASE 9 COMPLETE: Model-Ready Pipelines Assembled Successfully!")
print("Ready for Phase 10: Model Development!")
print("="*70)

FINAL MODEL-READY MATRICES VERIFICATION
X_train shape: (411040, 66)
X_test shape : (102761, 66)
Columns match perfectly: True
No NaNs in X_train: True
No NaNs in X_test : True

Saved files:
  - X_train_model_ready.csv
  - X_test_model_ready.csv
  - y_train_classification.csv
  - y_test_classification.csv
  - y_train_regression.csv
  - y_test_regression.csv

PHASE 9 COMPLETE: Model-Ready Pipelines Assembled Successfully!
Ready for Phase 10: Model Development!


In [8]:
# ---- Single final master dataset (train + test, fully processed) ----
final_master = pd.concat([X_train_final, X_test_final]).sort_index()

# Add back identifiers and both targets (aligned by original row index)
final_master.insert(0, "collision_index", df.loc[final_master.index, "collision_index"])
final_master.insert(1, "collision_year", df.loc[final_master.index, "collision_year"])
final_master["collision_severity"] = df.loc[final_master.index, "collision_severity"]
final_master["number_of_casualties"] = df.loc[final_master.index, "number_of_casualties"]

print("Shape:", final_master.shape)
print("Rows match original:", len(final_master) == len(df))
print("Total NaNs:", final_master.isna().sum().sum())

final_master.to_csv(data_path / "master_dataset_final.csv", index=False)
print("Saved: master_dataset_final.csv")


Shape: (513801, 70)
Rows match original: True
Total NaNs: 0
Saved: master_dataset_final.csv


In [10]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

data_path = Path("C:/Users/ACER/Downloads/New folder/cleaned_datasets")

# ---------- 1) Load ----------
df = pd.read_excel(data_path / "master_dataset_final.xlsx")   # slow (~2-3 min)
print("Loaded:", df.shape, "| NaNs:", df.isna().sum().sum())

# ---------- 2) month / hour_of_day -> cyclical sin/cos ----------
CYCLICAL = {"month": (1, 12), "hour_of_day": (0, 24)}   # col -> (lowest raw value, period)

for col, (raw_min, period) in CYCLICAL.items():
    uniq = np.sort(df[col].unique())
    assert len(uniq) == period, f"{col}: expected {period} unique values, got {len(uniq)}"
    steps = np.diff(uniq)
    assert np.allclose(steps, steps[0]), f"{col}: scaled values not evenly spaced"

    raw = np.round((df[col] - uniq[0]) / steps[0]).astype(int) + raw_min
    pos = df.columns.get_loc(col)
    df = df.drop(columns=[col])
    df.insert(pos,     f"{col}_sin", np.sin(2 * np.pi * raw / period))
    df.insert(pos + 1, f"{col}_cos", np.cos(2 * np.pi * raw / period))

# ---------- 3) Classification data: features + target ----------
# Drop identifiers and the regression target (number_of_casualties)
X = df.drop(columns=["collision_index", "collision_year", "collision_severity", "number_of_casualties"])
y = df["collision_severity"]

# ---------- 4) Stratified 70 / 15 / 15 split ----------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# ---------- 5) Verify ----------
for name, part in [("Train", X_train), ("Val", X_val), ("Test", X_test)]:
    print(f"{name:5s}: {len(part):>8,} rows ({len(part)/len(df):.1%})")
for name, t in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"  {name:5s} class %", (t.value_counts(normalize=True) * 100).round(2).sort_index().to_dict())
print("Columns identical:", list(X_train.columns) == list(X_val.columns) == list(X_test.columns))

# ---------- 6) Save (features and target together, one file per split) ----------
train = X_train.assign(collision_severity=y_train)
val   = X_val.assign(collision_severity=y_val)
test  = X_test.assign(collision_severity=y_test)

train.to_csv(data_path / "train_classification.csv", index=False)
val.to_csv(data_path / "validation_classification.csv", index=False)
test.to_csv(data_path / "test_classification.csv", index=False)
print("Saved: train_classification.csv, validation_classification.csv, test_classification.csv")


Loaded: (513793, 69) | NaNs: 0
Train:  359,655 rows (70.0%)
Val  :   77,069 rows (15.0%)
Test :   77,069 rows (15.0%)
  Train class % {1: 1.47, 2: 22.74, 3: 75.79}
  Val   class % {1: 1.47, 2: 22.73, 3: 75.8}
  Test  class % {1: 1.47, 2: 22.74, 3: 75.79}
Columns identical: True
Saved: train_classification.csv, validation_classification.csv, test_classification.csv
